# 08 — TMDB Movie × Keyword Valence Join

Joins `tmdb_movies_clean` to `tmdb_keyword_lexicon` to produce two tables:

1. **Exploded** — 1 row per (movie, keyword): all lexicon columns attached to each keyword instance
2. **Movie aggregate** — 1 row per movie: keywords treated as a single ngram, computing
   a movie-level "computed valence" from the distribution of its keyword valences

## Movie-level aggregate columns

| column | description |
|--------|-------------|
| `total_keywords` | total keyword count for this movie |
| `matched_keyword_count` | keywords with any valence score |
| `keyword_match_rate` | `matched / total` (0.0–1.0) |
| `avg_keyword_valence` | mean valence across matched keywords — the movie's "computed valence" |
| `min_keyword_valence` | most negative single keyword |
| `max_keyword_valence` | most positive single keyword |
| `valence_std` | spread of keyword sentiment (ddof=1) |
| `in_scl_count` | keywords with an exact SCL phrase match |
| `has_nma_count` | keywords with a negator/modal/adverb |
| `shift_strong_count` | keywords with `abs_composition_shift ≥ 0.30` |
| `avg_composition_shift` | mean shift across in_scl keywords |
| `max_abs_composition_shift` | strongest single shift in keyword set |
| `composition_type_dominant` | most common composition type in keyword set |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA    = Path('data')
OUT_DIR = Path('output')

## Load data

In [2]:
movies = pd.read_csv(
    OUT_DIR / 'tmdb-movies-clean/tmdb_movies_clean.csv',
    low_memory=False,
    usecols=['id', 'title', 'release_date', 'vote_average', 'vote_count',
             'popularity', 'genres', 'keywords'],
)
movies = movies[movies['keywords'].notna() & (movies['keywords'] != '')]
print(f'Movies with keywords: {len(movies):,}')
movies.head(3)

Movies with keywords: 283,875


,id,title,vote_average,vote_count,release_date,popularity,genres,keywords
0,565770,Blue Beetle,7.139,1023,2023-08-16,2994.357,"Action, Science Fiction, Adventure","armor, superhero, family relationships, family..."
1,980489,Gran Turismo,8.068,702,2023-08-09,2680.593,"Action, Drama, Adventure","based on true story, racing, based on video ga..."
2,968051,The Nun II,6.545,365,2023-09-06,1692.778,"Horror, Mystery, Thriller","france, bullying, sequel, religion, demon, got..."


In [3]:
lexicon = pd.read_csv(
    OUT_DIR / 'keyword-lexicon/tmdb_keyword_lexicon.csv',
    low_memory=False,
    usecols=['keyword', 'valence', 'valence_scale', 'valence_source',
             'in_scl', 'scl_token_coverage',
             'has_nma', 'composition_shift', 'abs_composition_shift',
             'shift_zscore', 'shift_percentile',
             'shift_strong', 'shift_very_strong', 'composition_type'],
)
print(f'Keyword lexicon: {len(lexicon):,} rows')
print(f'Coverage: {lexicon["valence"].notna().sum():,} keywords with valence ({lexicon["valence"].notna().mean()*100:.1f}%)')
lexicon.head(3)

Keyword lexicon: 67,916 rows
Coverage: 38,511 keywords with valence (56.7%)


,keyword,in_scl,scl_token_coverage,valence,valence_scale,valence_source,has_nma,composition_shift,abs_composition_shift,shift_zscore,shift_percentile,shift_strong,shift_very_strong,composition_type
0,short film,False,1.0,0.2675,bipolar,computed,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,woman director,False,0.5,0.3760,bipolar,computed,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gay pornography,False,1.0,-0.1355,bipolar,computed,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Explode: 1 row per (movie, keyword)

In [4]:
# Parse comma-separated keywords → list
movies['keyword_list'] = movies['keywords'].str.split(',').apply(
    lambda ks: [k.strip().lower() for k in ks if k.strip()]
)

# Explode to 1 row per (movie, keyword)
exploded = (
    movies[['id', 'title', 'release_date', 'vote_average', 'vote_count',
             'popularity', 'genres', 'keyword_list']]
    .explode('keyword_list')
    .rename(columns={'keyword_list': 'keyword'})
    .reset_index(drop=True)
)

print(f'Exploded rows: {len(exploded):,}')
print(f'Unique movies:  {exploded["id"].nunique():,}')
print(f'Unique keywords: {exploded["keyword"].nunique():,}')

Exploded rows: 957,097
Unique movies:  283,773
Unique keywords: 63,725


In [5]:
# Join lexicon columns
kw_joined = exploded.merge(lexicon, on='keyword', how='left')

matched = kw_joined['valence'].notna()
print(f'Lexicon match rate: {matched.sum():,} / {len(kw_joined):,} ({matched.mean()*100:.1f}%)')
print(f'\nvalence_source breakdown:')
print(kw_joined['valence_source'].value_counts(dropna=False).head(8).to_string())

kw_joined.head(5)

Lexicon match rate: 787,342 / 957,097 (82.3%)

valence_source breakdown:
valence_source
vad         364267
computed    336178
NaN         169755
opp          46552
nma          40345


,id,title,release_date,vote_average,vote_count,popularity,genres,keyword,in_scl,scl_token_coverage,...,valence_scale,valence_source,has_nma,composition_shift,abs_composition_shift,shift_zscore,shift_percentile,shift_strong,shift_very_strong,composition_type
0,565770,Blue Beetle,2023-08-16,7.139,1023,2994.357,"Action, Science Fiction, Adventure",armor,True,1.0,...,bipolar,vad,False,0.000,0.000,0.179526,39.404218,False,False,direct
1,565770,Blue Beetle,2023-08-16,7.139,1023,2994.357,"Action, Science Fiction, Adventure",superhero,True,1.0,...,bipolar,vad,False,0.000,0.000,0.179526,39.404218,False,False,direct
2,565770,Blue Beetle,2023-08-16,7.139,1023,2994.357,"Action, Science Fiction, Adventure",family relationships,False,0.5,...,bipolar,computed,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,565770,Blue Beetle,2023-08-16,7.139,1023,2994.357,"Action, Science Fiction, Adventure",family,True,1.0,...,bipolar,vad,False,0.000,0.000,0.179526,39.404218,False,False,direct
4,565770,Blue Beetle,2023-08-16,7.139,1023,2994.357,"Action, Science Fiction, Adventure",high tech,True,1.0,...,bipolar,vad,False,-0.244,0.244,-1.425127,90.604569,False,False,direct


## Aggregate: 1 row per movie

In [6]:
def agg_movie(g):
    matched = g[g['valence'].notna()]
    n_total   = len(g)
    n_matched = len(matched)

    row = {
        'total_keywords':          n_total,
        'matched_keyword_count':   n_matched,
        'keyword_match_rate':      n_matched / n_total if n_total > 0 else 0.0,
    }

    if n_matched > 0:
        v = matched['valence']
        row['avg_keyword_valence']  = v.mean()
        row['min_keyword_valence']  = v.min()
        row['max_keyword_valence']  = v.max()
        row['valence_std']          = v.std(ddof=1) if n_matched > 1 else None
    else:
        row['avg_keyword_valence']  = None
        row['min_keyword_valence']  = None
        row['max_keyword_valence']  = None
        row['valence_std']          = None

    row['in_scl_count']    = g['in_scl'].fillna(False).sum()
    row['has_nma_count']   = g['has_nma'].fillna(False).sum()
    row['shift_strong_count'] = g['shift_strong'].fillna(False).sum()

    scl = g[g['in_scl'].fillna(False)]
    row['avg_composition_shift']     = scl['composition_shift'].mean() if len(scl) > 0 else None
    row['max_abs_composition_shift'] = scl['abs_composition_shift'].max() if len(scl) > 0 else None

    ct = g['composition_type'].dropna()
    row['composition_type_dominant'] = ct.mode().iloc[0] if len(ct) > 0 else None

    return pd.Series(row)

print('Aggregating...')
agg = (
    kw_joined
    .groupby(['id', 'title', 'release_date', 'vote_average', 'vote_count', 'popularity', 'genres'])
    .apply(agg_movie, include_groups=False)
    .reset_index()
)

print(f'Movie aggregate: {agg.shape}')
print(f'\navg_keyword_valence stats:')
print(agg['avg_keyword_valence'].describe().round(4).to_string())
print(f'\ncomposition_type_dominant breakdown:')
print(agg['composition_type_dominant'].value_counts(dropna=False).to_string())
agg.head(5)

Aggregating...


Movie aggregate: (246497, 20)

avg_keyword_valence stats:
count    229501.0000
mean          0.1332
std           0.3530
min          -1.0000
25%          -0.0850
50%           0.1927
75%           0.3760
max           1.0000

composition_type_dominant breakdown:
composition_type_dominant
direct               147572
NaN                   79654
opposing_polarity      8946
nma_unclassified       6524
idiomatic              3795
degree_modifier           4
negation                  2


,id,title,release_date,vote_average,vote_count,popularity,genres,total_keywords,matched_keyword_count,keyword_match_rate,avg_keyword_valence,min_keyword_valence,max_keyword_valence,valence_std,in_scl_count,has_nma_count,shift_strong_count,avg_composition_shift,max_abs_composition_shift,composition_type_dominant
0,2,Ariel,1988-10-21,7.100,262,8.155,"Drama, Comedy, Romance",7.0,5.0,0.714286,-0.30790,-0.889,0.219,0.489435,3,0,0,0.000000,0.000,nma_unclassified
1,3,Shadows in Paradise,1986-10-17,7.199,281,5.946,"Drama, Comedy, Romance",4.0,2.0,0.500000,-0.25250,-0.672,0.167,0.593263,2,0,0,0.000000,0.000,direct
2,5,Four Rooms,1995-12-09,5.784,2436,15.295,Comedy,12.0,10.0,0.833333,0.03270,-0.416,0.480,0.352105,7,0,0,-0.023286,0.163,direct
3,6,Judgment Night,1993-10-15,6.533,302,13.564,"Action, Crime, Thriller",6.0,4.0,0.666667,-0.25575,-0.700,0.089,0.330838,3,0,1,-0.237667,0.713,direct
4,8,Life in Loops (A Megacities RMX),2006-01-01,7.700,25,1.587,Documentary,1.0,0.0,0.000000,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN,NaN


## Examples

In [7]:
has_valence = agg[agg['avg_keyword_valence'].notna()]

print('Most positive keyword profiles (high avg_keyword_valence):')
print(has_valence.nlargest(8, 'avg_keyword_valence')
      [['title', 'release_date', 'avg_keyword_valence', 'matched_keyword_count', 'composition_type_dominant']]
      .to_string(index=False))

print()
print('Most negative keyword profiles:')
print(has_valence.nsmallest(8, 'avg_keyword_valence')
      [['title', 'release_date', 'avg_keyword_valence', 'matched_keyword_count', 'composition_type_dominant']]
      .to_string(index=False))

print()
print('Strongest composition shifts (movie keyword sets):')
print(agg.nlargest(8, 'max_abs_composition_shift')
      [['title', 'max_abs_composition_shift', 'avg_composition_shift', 'in_scl_count', 'composition_type_dominant']]
      .to_string(index=False))

Most positive keyword profiles (high avg_keyword_valence):
                        title release_date  avg_keyword_valence  matched_keyword_count composition_type_dominant
       Patisserie Coin De Rue   2011-02-11                  1.0                    1.0                    direct
              Drums of Tahiti   1954-04-23                  1.0                    1.0                    direct
                 Tiara Tahiti   1962-07-01                  1.0                    1.0                    direct
Une lubie de monsieur Fortune   2010-01-25                  1.0                    1.0                    direct
Curiosity, Adventure and Love   2016-07-01                  1.0                    1.0                    direct
             Go with Your Gut   2017-06-10                  1.0                    1.0                    direct
         Her Sound, Her Story   2018-05-11                  1.0                    1.0                       NaN
                     Paradise   2020-

## Save

In [8]:
LEXICON_DIR = DATA / 'lexicons'
LEXICON_DIR.mkdir(parents=True, exist_ok=True)

# Keyword-level (exploded + joined)
kw_out = LEXICON_DIR / 'movie_keyword_exploded.pkl'
kw_joined.to_pickle(kw_out)
print(f'Exploded  → {kw_out}  ({kw_out.stat().st_size / 1e6:.1f} MB)  {kw_joined.shape}')

# Movie aggregate
agg_out = LEXICON_DIR / 'movie_keyword_aggregate.pkl'
agg.to_pickle(agg_out)
print(f'Aggregate → {agg_out}  ({agg_out.stat().st_size / 1e6:.1f} MB)  {agg.shape}')
print(f'\nAggregate columns: {list(agg.columns)}')

Exploded  → data/lexicons/movie_keyword_exploded.pkl  (139.8 MB)  (957097, 21)
Aggregate → data/lexicons/movie_keyword_aggregate.pkl  (36.5 MB)  (246497, 20)

Aggregate columns: ['id', 'title', 'release_date', 'vote_average', 'vote_count', 'popularity', 'genres', 'total_keywords', 'matched_keyword_count', 'keyword_match_rate', 'avg_keyword_valence', 'min_keyword_valence', 'max_keyword_valence', 'valence_std', 'in_scl_count', 'has_nma_count', 'shift_strong_count', 'avg_composition_shift', 'max_abs_composition_shift', 'composition_type_dominant']
